[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Prasathn01/wattwise-ml/blob/main/notebooks/PBL3_Unit5.ipynb)

# WattWise: PBL-III (Unit 5), tree models and ensembles for next-hour peak-load prediction

**21CSC305P Machine Learning · Team 4 · SRM Institute of Science and Technology, Kattankulathur**

Same task as PBL 1: will the next hour be a peak hour (mean load ≥ τ = 1.596 kW)? The notebook rebuilds the PBL 1 data from the raw file, tunes CART, bagging, random forest, AdaBoost, gradient boosting and a GAM with time-series cross-validation on 2006–09, combines them with Bayesian model averaging and stacking, and evaluates once on the 2010 test year.

**How to run:** Google Colab → *Runtime → Run all*. Figures are saved to `figures/` and numbers to `results_unit5.json`.

## 0. Setup
Libraries: pandas, NumPy, scikit-learn and matplotlib. Random seed 42.

In [ ]:
import os, json, time, zipfile, urllib.request, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, SplineTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import (BaggingClassifier, RandomForestClassifier, AdaBoostClassifier,
                              GradientBoostingClassifier)
from sklearn.metrics import (roc_auc_score, accuracy_score, precision_score, recall_score, f1_score,
                             brier_score_loss, log_loss, roc_curve)
from sklearn.inspection import permutation_importance
warnings.filterwarnings("ignore")
SEED = 42
FIG = "figures"; os.makedirs(FIG, exist_ok=True)
RESULTS = {}
COLORS = ["#2F6FB0", "#E0892B", "#2F855A", "#C53030", "#6B46C1", "#975A16", "#B83280", "#4A5568"]
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False})
def savefig(name):
    plt.tight_layout(); plt.savefig(f"{FIG}/{name}.png", dpi=150, bbox_inches="tight"); plt.show()

## 1. Load the data and make hourly means
The UCI file is downloaded if it is not present. Hourly means; gaps of up to 3 hours are time-interpolated; the remaining outage hours are dropped.

In [ ]:
DATA_URL = "https://archive.ics.uci.edu/static/public/235/individual+household+electric+power+consumption.zip"
DATA_TXT = "household_power_consumption.txt"
if not os.path.exists(DATA_TXT):
    urllib.request.urlretrieve(DATA_URL, "hpc.zip")
    with zipfile.ZipFile("hpc.zip") as z:
        z.extract(DATA_TXT)
raw = pd.read_csv(DATA_TXT, sep=";", na_values="?", low_memory=False)
raw.index = pd.to_datetime(raw["Date"] + " " + raw["Time"], format="%d/%m/%Y %H:%M:%S")
raw = raw.drop(columns=["Date", "Time"]).astype(float)
print(f"{len(raw):,} minute readings, {raw['Global_active_power'].isna().mean():.2%} missing")
h = raw.resample("h").mean()
hi = h.interpolate(method="time", limit=3, limit_area="inside")
y_kw = hi["Global_active_power"]
print(f"Hourly means: {len(h):,}; dropped outage hours: {y_kw.isna().sum()}")

## 2. The 37 PBL 1 features, the peak label and the chronological split
Same features, threshold (75th percentile of 2006–09 hours) and split as PBL 1. The persistence RMSE check must match PBL 1 (0.618 kW).

In [ ]:
X = pd.DataFrame(index=hi.index)
for L in [1, 2, 3, 24, 168]:
    X[f"lag{L}"] = y_kw.shift(L)
X["mean_prev24"] = y_kw.shift(1).rolling(24).mean()
for col, name in [("Voltage", "voltage_prev"), ("Global_reactive_power", "reactive_prev"),
                  ("Sub_metering_1", "kitchen_prev"), ("Sub_metering_2", "laundry_prev"), ("Sub_metering_3", "heater_ac_prev")]:
    X[name] = hi[col].shift(1)
for k in range(1, 24):
    X[f"hour_{k:02d}"] = (hi.index.hour == k).astype(float)
X["weekend"] = (hi.index.dayofweek >= 5).astype(float)
X["month_sin"] = np.sin(2 * np.pi * hi.index.month / 12)
X["month_cos"] = np.cos(2 * np.pi * hi.index.month / 12)
X["power"] = y_kw
data = X.dropna()
FEATS = [c for c in data.columns if c != "power"]
TAU = float(np.quantile(y_kw[y_kw.index < "2010-01-01"].dropna(), 0.75))
data["peak"] = (data["power"] >= TAU).astype(int)
train = data[data.index < "2010-01-01"]; test = data[data.index >= "2010-01-01"]
Xtr, ytr = train[FEATS].values, train["peak"].values
Xte, yte = test[FEATS].values, test["peak"].values
print(f"Features: {len(FEATS)} | train {len(train):,} h ({train.index[0]:%d %b %Y} – {train.index[-1]:%d %b %Y}) | test {len(test):,} h")
print(f"Peak threshold tau = {TAU:.3f} kW | peak share: train {ytr.mean():.1%}, test {yte.mean():.1%}")
pers = test["lag1"].values
print(f"Check against PBL 1 – persistence RMSE {np.sqrt(np.mean((test['power'] - pers) ** 2)):.3f} kW (PBL 1: 0.618)")
RESULTS.update(n_train=len(train), n_test=len(test), tau_kW=round(TAU, 3), peak_share_train=round(float(ytr.mean()), 4), peak_share_test=round(float(yte.mean()), 4))

## 3. Feature analysis for trees
Peak rate by hour of day and by decile of the previous hour's load.

In [ ]:
rate_hour = train.groupby(train.index.hour)["peak"].mean()
dec = pd.qcut(train["lag1"], 10, labels=False)
rate_lag = train.groupby(dec)["peak"].mean()
lag_edges = train.groupby(dec)["lag1"].max()
print("Peak rate by previous-hour load decile:", (100 * rate_lag).round(1).tolist())
print(f"Lowest-decile upper edge {lag_edges.iloc[0]:.2f} kW; top-decile lower edge {lag_edges.iloc[8]:.2f} kW")
RESULTS["peak_rate_by_hour"] = (100 * rate_hour).round(1).tolist()
RESULTS["peak_rate_by_lag1_decile"] = (100 * rate_lag).round(1).tolist()
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].bar(rate_hour.index, 100 * rate_hour.values, color=COLORS[0]); ax[0].axhline(100 * ytr.mean(), ls="--", color="grey")
ax[0].set(xlabel="hour of day", ylabel="% of hours that are peaks", title="Peak rate by hour of day (training years)")
ax[1].bar(range(1, 11), 100 * rate_lag.values, color=COLORS[1]); ax[1].axhline(100 * ytr.mean(), ls="--", color="grey")
ax[1].set(xlabel="decile of previous-hour load (1 = lowest)", ylabel="% of next hours that are peaks", title="Peak rate by previous-hour load")
savefig("u5_feature_target")

## 4. Model selection helpers
`TimeSeriesSplit(4)` on the training years only; the 2010 test year is used once.

In [ ]:
tscv = TimeSeriesSplit(n_splits=4)
def cv_auc(make_model, Xa=Xtr):
    s = []
    for a, b in tscv.split(Xa):
        m = make_model().fit(Xa[a], ytr[a])
        s.append(roc_auc_score(ytr[b], m.predict_proba(Xa[b])[:, 1]))
    return float(np.mean(s))
def tune(name, grid, make):
    rows = []
    for params in grid:
        t = time.time(); auc = cv_auc(lambda: make(**params))
        rows.append({**params, "cv_auc": round(auc, 4), "sec": round(time.time() - t, 1)})
    df = pd.DataFrame(rows); best = df.loc[df["cv_auc"].idxmax()]
    print(f"{name}:"); print(df.to_string(index=False))
    params = {k: (int(best[k]) if isinstance(grid[0][k], (int, np.integer)) else best[k]) for k in grid[0]}
    print("-> chosen", params, "\n")
    RESULTS.setdefault("cv", {})[name] = df.to_dict(orient="records")
    return params
def metrics(p, thr=0.5):
    yhat = (p >= thr).astype(int)
    return {"accuracy": accuracy_score(yte, yhat), "precision": precision_score(yte, yhat, zero_division=0),
            "recall": recall_score(yte, yhat), "F1": f1_score(yte, yhat), "ROC_AUC": roc_auc_score(yte, p),
            "Brier": brier_score_loss(yte, p), "log_loss": log_loss(yte, np.clip(p, 1e-6, 1 - 1e-6))}
FINAL, PROBA, FIT_SEC = {}, {}, {}
def fit_final(name, model, Xa=Xtr, Xb=Xte):
    t = time.time(); model.fit(Xa, ytr); FIT_SEC[name] = time.time() - t
    FINAL[name] = model; PROBA[name] = model.predict_proba(Xb)[:, 1]
    return model

## 5. Reference: logistic regression (PBL 1 model)

In [ ]:
fit_final("Logistic regression (PBL 1 reference)", make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=2000)))
print(f"Logistic regression test ROC-AUC: {roc_auc_score(yte, PROBA['Logistic regression (PBL 1 reference)']):.3f}")

## 6. CART: classification tree with cost-complexity pruning

In [ ]:
grid = [{"max_depth": d, "min_samples_leaf": l} for d in [4, 6, 8, 12] for l in [20, 100]]
p_cart = tune("CART", grid, lambda **k: DecisionTreeClassifier(random_state=SEED, **k))
# cost-complexity pruning path on the chosen tree
base = DecisionTreeClassifier(random_state=SEED, **p_cart)
alphas = base.cost_complexity_pruning_path(Xtr, ytr).ccp_alphas
alphas = np.unique(np.quantile(alphas, [0, 0.25, 0.5, 0.75, 0.9]))
pr = [{"ccp_alpha": float(a)} for a in alphas]
p_alpha = tune("CART pruning", pr, lambda **k: DecisionTreeClassifier(random_state=SEED, **p_cart, **k))
cart = fit_final("CART", DecisionTreeClassifier(random_state=SEED, **p_cart, **p_alpha))
print(f"Final tree: depth {cart.get_depth()}, {cart.get_n_leaves()} leaves")
RESULTS["cart"] = {**p_cart, **p_alpha, "depth": int(cart.get_depth()), "leaves": int(cart.get_n_leaves())}
small = DecisionTreeClassifier(max_depth=3, random_state=SEED).fit(Xtr, ytr)
plt.figure(figsize=(18, 7))
plot_tree(small, feature_names=FEATS, class_names=["off-peak", "peak"], filled=True, impurity=True, proportion=True, rounded=True, fontsize=10)
plt.title("CART tree limited to depth 3 (for reading; the tuned tree is deeper)")
savefig("u5_cart_tree_depth3")

## 7. Bagging

In [ ]:
bag = fit_final("Bagging (200 trees)", BaggingClassifier(DecisionTreeClassifier(min_samples_leaf=20, random_state=SEED),
                                                       n_estimators=200, random_state=SEED, n_jobs=-1))

## 8. Random forest

In [ ]:
grid = [{"max_features": f, "min_samples_leaf": l} for f in [0.2, 0.4] for l in [5, 20]]
p_rf = tune("Random forest", grid, lambda **k: RandomForestClassifier(n_estimators=150, random_state=SEED, n_jobs=-1, **k))
rf = fit_final("Random forest", RandomForestClassifier(n_estimators=400, random_state=SEED, n_jobs=-1, **p_rf))
RESULTS["random_forest"] = {**p_rf, "n_estimators": 400}

## 9. AdaBoost

In [ ]:
grid = [{"n_estimators": n, "learning_rate": lr} for n in [200, 400] for lr in [0.1, 0.5]]
p_ada = tune("AdaBoost", grid, lambda **k: AdaBoostClassifier(DecisionTreeClassifier(max_depth=2, random_state=SEED), random_state=SEED, **k))
ada = fit_final("AdaBoost", AdaBoostClassifier(DecisionTreeClassifier(max_depth=2, random_state=SEED), random_state=SEED, **p_ada))
RESULTS["adaboost"] = {**p_ada, "base": "depth-2 tree"}

## 10. Gradient boosting

In [ ]:
grid = [{"max_depth": d, "learning_rate": lr} for d in [2, 3] for lr in [0.05, 0.1]]
p_gb = tune("Gradient boosting", grid, lambda **k: GradientBoostingClassifier(n_estimators=300, subsample=0.8, random_state=SEED, **k))
gb = fit_final("Gradient boosting", GradientBoostingClassifier(n_estimators=300, subsample=0.8, random_state=SEED, **p_gb))
RESULTS["gradient_boosting"] = {**p_gb, "n_estimators": 300, "subsample": 0.8}
# test log-loss as boosting rounds are added (shows how the additive model builds up)
ll_gb = [log_loss(yte, np.clip(p[:, 1], 1e-6, 1 - 1e-6)) for p in gb.staged_predict_proba(Xte)]
ll_ada = [log_loss(yte, np.clip(p[:, 1], 1e-6, 1 - 1e-6)) for p in ada.staged_predict_proba(Xte)]
plt.figure(figsize=(9, 4))
plt.plot(range(1, len(ll_gb) + 1), ll_gb, color=COLORS[0], label="gradient boosting")
plt.plot(range(1, len(ll_ada) + 1), ll_ada, color=COLORS[1], label="AdaBoost")
plt.xlabel("number of trees (boosting rounds)"); plt.ylabel("test log-loss"); plt.legend(); plt.title("Boosting: test log-loss as trees are added")
savefig("u5_boosting_rounds")
RESULTS["boosting_rounds"] = {"gb_best_round": int(np.argmin(ll_gb) + 1), "ada_best_round": int(np.argmin(ll_ada) + 1)}

## 11. Generalized additive model (logistic, cubic splines)

In [ ]:
CONT = ["lag1", "lag2", "lag3", "lag24", "lag168", "mean_prev24", "voltage_prev", "reactive_prev", "kitchen_prev", "laundry_prev", "heater_ac_prev"]
ci = [FEATS.index(c) for c in CONT]; oi = [i for i in range(len(FEATS)) if i not in ci]
def make_gam(C=1.0):
    pre = ColumnTransformer([("spl", SplineTransformer(n_knots=6, degree=3), ci), ("lin", "passthrough", oi)])
    return make_pipeline(pre, StandardScaler(), LogisticRegression(C=C, max_iter=3000))
p_gam = tune("GAM", [{"C": c} for c in [0.01, 0.1, 1.0]], make_gam)
gam = fit_final("GAM (logistic, splines)", make_gam(**p_gam))
RESULTS["gam"] = {**p_gam, "n_knots": 6, "degree": 3}
# partial effect of previous-hour load: vary lag1, hold other features at their training median
grid_l1 = np.linspace(np.quantile(Xtr[:, 0], 0.01), np.quantile(Xtr[:, 0], 0.99), 60)
Xm = np.tile(np.median(Xtr, axis=0), (60, 1)); Xm[:, 0] = grid_l1
eff = gam.decision_function(Xm)
# partial effect of hour: one-hot hour columns at median profile
hour_eff = []
for hr in range(24):
    xr = np.median(Xtr, axis=0).copy()
    for k in range(1, 24): xr[FEATS.index(f"hour_{k:02d}")] = 1.0 if k == hr else 0.0
    hour_eff.append(gam.decision_function(xr[None, :])[0])
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(grid_l1, eff, color=COLORS[0], lw=2.5); ax[0].set(xlabel="previous-hour load (kW)", ylabel="log-odds of peak", title="GAM: effect of previous-hour load")
ax[1].plot(range(24), hour_eff, "o-", color=COLORS[1]); ax[1].set(xlabel="hour of day", ylabel="log-odds of peak", title="GAM: effect of hour of day")
savefig("u5_gam_effects")

## 12. Bayesian model averaging and stacking
Both are fitted on the last 20% of the training years.

In [ ]:
cut = int(0.8 * len(Xtr)); Xa, ya, Xb, yb = Xtr[:cut], ytr[:cut], Xtr[cut:], ytr[cut:]
from sklearn.base import clone
members = ["Logistic regression (PBL 1 reference)", "CART", "Random forest", "AdaBoost", "Gradient boosting", "GAM (logistic, splines)"]
P_hold = {}
for m in members:
    mm = clone(FINAL[m]).fit(Xa, ya); P_hold[m] = np.clip(mm.predict_proba(Xb)[:, 1], 1e-6, 1 - 1e-6)
LL = {m: float(np.sum(yb * np.log(P_hold[m]) + (1 - yb) * np.log(1 - P_hold[m]))) for m in members}
mx = max(LL.values()); w = {m: np.exp(LL[m] - mx) for m in members}; s = sum(w.values()); w = {m: w[m] / s for m in members}
print("Held-out log-likelihood and BMA posterior weight:")
for m in members: print(f"  {m:40s} LL = {LL[m]:10.1f}   weight = {w[m]:.4f}")
PROBA["BMA"] = sum(w[m] * PROBA[m] for m in members)
avg = np.mean([PROBA[m] for m in members], axis=0); PROBA["Simple average"] = avg
meta = LogisticRegression(max_iter=2000).fit(np.column_stack([np.log(P_hold[m] / (1 - P_hold[m])) for m in members]), yb)
Zte = np.column_stack([np.log(np.clip(PROBA[m], 1e-6, 1 - 1e-6) / (1 - np.clip(PROBA[m], 1e-6, 1 - 1e-6))) for m in members])
PROBA["Stacking"] = meta.predict_proba(Zte)[:, 1]
print("Stacking weights (on log-odds):", dict(zip(members, meta.coef_[0].round(3))))
RESULTS["bma_weights"] = {m: round(float(w[m]), 4) for m in members}
RESULTS["holdout_loglik"] = {m: round(LL[m], 1) for m in members}
RESULTS["stacking_coef"] = {m: round(float(c), 3) for m, c in zip(members, meta.coef_[0])}

## 13. Comparison on the 2010 test year

In [ ]:
order = ["Logistic regression (PBL 1 reference)", "CART", "Bagging (200 trees)", "Random forest", "AdaBoost",
         "Gradient boosting", "GAM (logistic, splines)", "Simple average", "BMA", "Stacking"]
rows = []
for m in order:
    r = metrics(PROBA[m]); r["fit_sec"] = round(FIT_SEC.get(m, np.nan), 1); rows.append({"model": m, **r})
comp = pd.DataFrame(rows).round(3)
print(comp.to_string(index=False))
RESULTS["test"] = comp.to_dict(orient="records")
plt.figure(figsize=(7, 6))
for i, m in enumerate(["Logistic regression (PBL 1 reference)", "CART", "Random forest", "AdaBoost", "Gradient boosting", "GAM (logistic, splines)", "Stacking"]):
    fpr, tpr, _ = roc_curve(yte, PROBA[m]); plt.plot(fpr, tpr, color=COLORS[i % 8], lw=1.8, label=f"{m.split(' (')[0]} ({roc_auc_score(yte, PROBA[m]):.3f})")
plt.plot([0, 1], [0, 1], ls="--", color="grey"); plt.xlabel("false-positive rate"); plt.ylabel("true-positive rate"); plt.title("ROC curves on the 2010 test year"); plt.legend(fontsize=8)
savefig("u5_roc")

## 14. Feature importance

In [ ]:
imp_rf = pd.Series(rf.feature_importances_, index=FEATS).sort_values(ascending=False)
perm = permutation_importance(gb, Xte, yte, scoring="roc_auc", n_repeats=5, random_state=SEED, n_jobs=-1)
imp_gb = pd.Series(perm.importances_mean, index=FEATS).sort_values(ascending=False)
print("Random forest (impurity) top 8:\n", imp_rf.head(8).round(3).to_string())
print("\nGradient boosting (permutation, drop in test AUC) top 8:\n", imp_gb.head(8).round(4).to_string())
RESULTS["importance_rf_top8"] = imp_rf.head(8).round(4).to_dict(); RESULTS["importance_gb_perm_top8"] = imp_gb.head(8).round(4).to_dict()
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
imp_rf.head(10)[::-1].plot.barh(ax=ax[0], color=COLORS[0]); ax[0].set(title="Random forest: impurity importance", xlabel="mean decrease in Gini")
imp_gb.head(10)[::-1].plot.barh(ax=ax[1], color=COLORS[1]); ax[1].set(title="Gradient boosting: permutation importance (test)", xlabel="drop in ROC-AUC")
savefig("u5_importance")

## 15. Decision threshold and saved results
The threshold is chosen on the held-out block of the training years, not on 2010.

In [ ]:
# model chosen by its cross-validated AUC on the training years (not by the test year)
cv_name = {"CART pruning": "CART", "Random forest": "Random forest", "AdaBoost": "AdaBoost",
           "Gradient boosting": "Gradient boosting", "GAM": "GAM (logistic, splines)"}
cv_best = {cv_name[k]: max(r["cv_auc"] for r in v) for k, v in RESULTS["cv"].items() if k in cv_name}
best_name = max(cv_best, key=cv_best.get)
mm = clone(FINAL[best_name]).fit(Xa, ya); ph = mm.predict_proba(Xb)[:, 1]
ths = np.linspace(0.1, 0.7, 61); f1s = [f1_score(yb, (ph >= t).astype(int)) for t in ths]
t_star = float(ths[int(np.argmax(f1s))])
r05, rt = metrics(PROBA[best_name], 0.5), metrics(PROBA[best_name], t_star)
print(f"Best single model by cross-validated AUC: {best_name} ({cv_best[best_name]:.3f}); F1-optimal threshold on held-out block = {t_star:.2f}")
print(f"  threshold 0.50: precision {r05['precision']:.3f}, recall {r05['recall']:.3f}, F1 {r05['F1']:.3f}")
print(f"  threshold {t_star:.2f}: precision {rt['precision']:.3f}, recall {rt['recall']:.3f}, F1 {rt['F1']:.3f}")
RESULTS["threshold"] = {"model": best_name, "t_star": round(t_star, 2), "at_0.5": {k: round(v, 3) for k, v in r05.items()}, "at_t_star": {k: round(v, 3) for k, v in rt.items()}}
with open("results_unit5.json", "w") as f:
    json.dump(RESULTS, f, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))
print("Saved results_unit5.json and figures/")